# ml-boilerplate as a library — à la carte examples

**Setup (pick one, then restart kernel if you just installed):**
```bash
pip install "ml-boilerplate @ git+https://github.com/rajindersingh041/ml_boiler_plate.git"
# or: uv pip install "ml-boilerplate @ git+https://github.com/rajindersingh041/ml_boiler_plate.git"
# or: git pull this repo and work next to data/ (CSVs vendored, runs offline)
```
Every cell below runs top-to-bottom with **no YAML, no CLI** — plain function calls.
Data: `data/*.csv` if present (offline), else built-in synthetic generators.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline

from ml_boilerplate import registry
from ml_boilerplate.config import Config, ModelConfig
from ml_boilerplate.crossval import run_cv
from ml_boilerplate.data import load_data
from ml_boilerplate.distributions import data_distributions, error_distributions
from ml_boilerplate.eda import profile_dataframe
from ml_boilerplate.engineering import add_datetime_features, add_lag_features, featurize
from ml_boilerplate.evaluation import evaluate_on_test
from ml_boilerplate.io import read_table
from ml_boilerplate.model import build_model
from ml_boilerplate.monitor import check_drift
from ml_boilerplate.tasks import get_task
from ml_boilerplate.validation import split_train_val_test

print('imports ok')

# vendored CSVs live in <repo>/data no matter where the kernel was started
DATA = next((p / 'data' for p in [Path.cwd(), *Path.cwd().parents]
               if (p / 'data' / 'mpg.csv').exists()), Path('data'))
print('data dir:', DATA.resolve())

## 1. Load data — file, URL, or synthetic (no config file needed)

In [ ]:
try:
    df = read_table(str(DATA / 'mpg.csv'))
    print('vendored mpg.csv')
except FileNotFoundError:
    cfg0 = Config(task='regression')
    df = load_data(cfg0.data, 'regression').rename(columns={'target': 'mpg'})
    print('synthetic fallback (target renamed to mpg)')
print(df.shape)
display(df.head(3))

## 2. Profile it — EDA dict + distributions (JSON-serializable, plottable)

In [ ]:
prof = profile_dataframe(df)
print('rows/cols:', prof['n_rows'], prof['n_columns'])
print('missing:', {k: v for k, v in prof['missing'].items() if v['count']})

dists = data_distributions(df, 'mpg')
print('dist keys:', sorted(dists.keys()))
print('mpg mean: %.1f' % np.mean(dists['histograms']['mpg']))

## 3. Engineer features — datetime parts + holidays, lags + rolling (timeseries)

In [ ]:
# 3a. datetime parts on any date column (row-local, leak-free)
try:
    sun = read_table(str(DATA / 'monthly-sunspots.csv'))
except FileNotFoundError:
    sun = pd.DataFrame({'Month': pd.date_range('2000-01-01', periods=24, freq='ME').astype(str),
                        'Sunspots': np.linspace(10, 50, 24)})
    print('synthetic sunspots fallback')
sun2 = add_datetime_features(sun, 'Month', cyclical=True, holiday_country='US')
print([c for c in sun2.columns if c.startswith('Month_')][:6], '...')

# 3b. lags + rolling on a target series (shift-1 rule: no leakage)
ts = pd.DataFrame({'date': pd.date_range('2024-01-01', periods=30, freq='D'),
                     'y': np.arange(30, dtype=float)})
lagged = add_lag_features(ts, target='y', date_col='date', n_lags=[1, 7],
                            rolling_windows=[7], rolling_stats=['mean', 'max'],
                            lag_diffs=True)
print(lagged.shape, 'warm-up rows dropped:', 30 - len(lagged))
display(lagged.head(3))

## 4. Split 3 ways — stratified / chronological handled by task name

In [ ]:
cfg = Config(task='regression')
cfg.data.test_size = 0.2
cfg.validation.val_size = 0.2
cfg.data.random_state = 42

X = df.drop(columns=['mpg'])
y = df['mpg']
Xtr, Xv, Xte, ytr, yv, yte = split_train_val_test(X, y, cfg, 'regression')
print(f'train={len(Xtr)}, val={len(Xv)}, test={len(Xte)}')

## 5. Preprocess + model — same pipeline train and serve share

In [ ]:
task = get_task('regression')
pipe = Pipeline([
    ('preprocess', task.build_preprocessor(Xtr, cfg)),
    ('model', build_model(ModelConfig(type='random_forest', params={'n_estimators': 50}),
                           task.model_registry())),
])
pipe.fit(Xtr, ytr)
pred = pipe.predict(Xte)
print('R2 on test: %.3f' % pipe.score(Xte, yte))

## 6. Cross-validate + gate the release

In [ ]:
cfg.cross_validation.folds = 3
cv = run_cv(Xtr, ytr, cfg, 'regression')
print('CV R2: %.3f ± %.3f' % (cv['aggregate']['r2']['mean'], cv['aggregate']['r2']['std']))

gate = evaluate_on_test(yte.to_numpy(), pred, None, 'regression', {'r2': 0.5})
print('gate passed:', gate['passed'], '| failed:', gate['failed'])

err = error_distributions(yte.to_numpy(), pred, None, 'regression')
print('residual keys:', sorted(err.keys()))

## 7. Register, promote, serve-check, monitor drift

In [ ]:
tmp = Path(tempfile.mkdtemp())
run_id = registry.save_run(str(tmp / 'runs'), pipe,
                             metrics={'r2': gate['metrics']['r2']},
                             params={'n_estimators': 50},
                             profile=data_distributions(df.assign(mpg=y), 'mpg'),
                             cfg_snapshot={'task': 'regression'})
registry.promote(str(tmp / 'runs'), str(tmp / 'prod'), run_id)
loaded = registry.load_production(str(tmp / 'prod'))
print('promoted', run_id, '| loaded predicts:', len(loaded.predict(Xte)))

# drift: same data -> no drift; shifted data -> drift
profile = data_distributions(df, 'mpg')
print('no drift:', check_drift(profile, df, psi_threshold=0.25)['drifted'])
shifted = df.copy(); shifted['weight'] = shifted['weight'] * 3
print('shifted weight drift:', check_drift(profile, shifted, psi_threshold=0.25)['drifted'])

## Cheat sheet — what to import for what

| Need | Import |
|---|---| 
| Read csv/parquet, local or URL | `from ml_boilerplate.io import read_table, write_table` |
| Synthetic data (offline/tests) | `from ml_boilerplate.data import load_data` |
| EDA profile dict | `from ml_boilerplate.eda import profile_dataframe, run_eda` |
| Datetime / lag / rolling features | `from ml_boilerplate.engineering import add_datetime_features, add_lag_features, featurize` |
| Train/val/test split | `from ml_boilerplate.validation import split_train_val_test` |
| Shared preprocessor | `task.build_preprocessor(X, cfg)` via `get_task(name)` |
| Model by name | `from ml_boilerplate.model import build_model` + registries |
| K-fold CV | `from ml_boilerplate.crossval import run_cv` |
| Grid/random tuning | `from ml_boilerplate.tuning import run_tuning` |
| Test gate | `from ml_boilerplate.evaluation import evaluate_on_test` |
| Save/promote/load model | `from ml_boilerplate import registry` |
| Drift check | `from ml_boilerplate.monitor import check_drift` |
| Serve over HTTP | `from ml_boilerplate.api import create_app` |

`Config(task=...)` built in code replaces every YAML file; only `run_training`/`main` need YAML.